# 🦀 Day 2: Move Semantics — `Copy` vs `Clone`

Today:
- What moves, what copies
- The `Copy` trait
- The `Clone` trait
- Intentionally triggering and fixing move errors

---

## 📋 Copy vs Clone

| | `Copy` | `Clone` |
|---|---|---|
| When | Implicit (on assignment) | Explicit (`.clone()`) |
| Cost | Cheap (bitwise copy) | Potentially expensive (deep copy) |
| Types | Simple stack types | Any type that implements it |
| Requirement | Type must be fully on the stack | Just needs implementation |

In [ ]:
// Copy types — implicit bitwise copy on assignment
let a: i32 = 10;
let b = a;   // Copy — both valid

let x: (i32, f64) = (1, 2.0);
let y = x;   // Copy — tuples of Copy types are Copy

let arr = [1, 2, 3];
let arr2 = arr;  // Copy — arrays of Copy types are Copy

println!("All still valid: a={}, b={}, x={:?}, y={:?}, arr={:?}, arr2={:?}",
    a, b, x, y, arr, arr2);

In [ ]:
// Clone — explicit deep copy
let s1 = String::from("hello");
let s2 = s1.clone();  // Deep copy — both valid!
println!("s1={}, s2={}", s1, s2);

let v1 = vec![1, 2, 3];
let v2 = v1.clone();  // Deep copy of the entire vector
println!("v1={:?}, v2={:?}", v1, v2);

// Cloning is explicit because it can be expensive!
let big = vec![0u8; 1_000_000]; // 1MB vector
let big_copy = big.clone(); // Allocates another 1MB!
println!("Both have {} bytes", big.len());

## 🏗️ Making Your Types `Copy` and `Clone`

In [ ]:
// Deriving Copy + Clone (only works if ALL fields are Copy)
#[derive(Debug, Copy, Clone)]
struct Point {
    x: f64,
    y: f64,
}

let p1 = Point { x: 1.0, y: 2.0 };
let p2 = p1;  // Copy! Both valid
println!("p1={:?}, p2={:?}", p1, p2);

// Can't derive Copy if a field is not Copy
#[derive(Debug, Clone)]  // Only Clone, not Copy!
struct Person {
    name: String,    // String is NOT Copy
    age: u32,        // u32 IS Copy
}

let p1 = Person { name: "Alice".into(), age: 30 };
let p2 = p1.clone();  // Must use clone()
// let p3 = p1;        // This would MOVE p1
println!("p1={:?}, p2={:?}", p1, p2);

In [ ]:
// Enums follow the same rules
#[derive(Debug, Copy, Clone)]
enum Color {
    Red,
    Green,
    Blue,
    Custom(u8, u8, u8),  // All fields are Copy
}

let c1 = Color::Custom(255, 128, 0);
let c2 = c1;  // Copy!
println!("c1={:?}, c2={:?}", c1, c2);

#[derive(Debug, Clone)]  // Can't be Copy — contains String
enum Message {
    Text(String),
    Number(i32),
}

let m1 = Message::Text("hello".into());
let m2 = m1.clone();
println!("m1={:?}, m2={:?}", m1, m2);

## 🐛 Common Move Errors & Fixes

In [ ]:
// Error 1: Using a value after move
// FIX: Clone before moving
let name = String::from("Alice");
let greeting = format!("Hello, {}!", name); // format! borrows, doesn't move!
println!("{} — name is still: {}", greeting, name); // ✅

// But + operator MOVES the left operand:
let first = String::from("Hello");
let second = String::from(" World");
let combined = first + &second;  // first is MOVED
// println!("{}", first);  // ❌ first was moved
println!("{}", combined);   // ✅
println!("{}", second);     // ✅ (was only borrowed)

In [ ]:
// Error 2: Moving in a loop
let data = String::from("important");

// BAD: This would move data on first iteration
// for _ in 0..3 {
//     let copy = data;  // ❌ moved on first iteration!
// }

// FIX 1: Clone each iteration
for i in 0..3 {
    let copy = data.clone();
    println!("Iteration {}: {}", i, copy);
}
println!("Original still valid: {}", data);

// FIX 2: Use a reference (better!)
for i in 0..3 {
    println!("Iteration {}: {}", i, &data);
}

In [ ]:
// Error 3: Partial move from struct
#[derive(Debug)]
struct User {
    name: String,
    email: String,
    age: u32,
}

let user = User {
    name: "Alice".into(),
    email: "alice@email.com".into(),
    age: 30,
};

let name = user.name;  // Partial move! name field is moved out
println!("Name: {}", name);
println!("Age: {}", user.age);   // ✅ age is Copy, still accessible
// println!("User: {:?}", user);  // ❌ Can't use whole struct
// println!("Email: {}", user.email); // ✅ email wasn't moved

## 📊 Move Semantics Cheat Sheet

In [ ]:
println!("╔════════════════════════════════════════════╗");
println!("║     MOVE vs COPY CHEAT SHEET              ║");
println!("╠════════════════════════════════════════════╣");
println!("║ COPIES (implicit):                        ║");
println!("║   i8-i128, u8-u128, f32, f64             ║");
println!("║   bool, char                              ║");
println!("║   (T, U) if T and U are Copy              ║");
println!("║   [T; N] if T is Copy                     ║");
println!("║   &T (references are always Copy)         ║");
println!("╠════════════════════════════════════════════╣");
println!("║ MOVES (invalidates source):               ║");
println!("║   String, Vec<T>, HashMap<K,V>            ║");
println!("║   Box<T>, any heap-allocated type          ║");
println!("║   Structs/enums with non-Copy fields      ║");
println!("╠════════════════════════════════════════════╣");
println!("║ TO AVOID MOVING:                          ║");
println!("║   1. Use .clone() (deep copy)             ║");
println!("║   2. Use & references (borrow)            ║");
println!("║   3. Derive Copy if all fields are Copy   ║");
println!("╚════════════════════════════════════════════╝");

## 🏋️ Exercises

In [ ]:
// Exercise 1: Fix this code WITHOUT using clone()
// Hint: change the function signature
/*
fn print_length(s: String) {
    println!("Length: {}", s.len());
}

let msg = String::from("hello world");
print_length(msg);
println!("Message: {}", msg);  // Error!
*/

// YOUR CODE HERE


In [ ]:
// Exercise 2: Create a struct `Dimensions` with width, height (f64)
// that is Copy. Then create a `Box3D` struct with Dimensions + depth
// that is also Copy. Prove they copy correctly.

// YOUR CODE HERE


## 🎯 Key Takeaways

1. `Copy` is implicit and cheap (stack-only types)
2. `Clone` is explicit and can be expensive (heap allocations)
3. You can `#[derive(Copy, Clone)]` only if **all fields** are `Copy`
4. References (`&T`) are always `Copy` — this is key for borrowing!
5. When in doubt: borrow (`&`) rather than clone

---
**Tomorrow:** References & borrowing! 📎